In [1]:
import pyshacl
from pyshacl import validate
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, RDFS, XSD
import sys


In [2]:
# Check ttl syntax
rdf_Graph=Graph()

try:
    rdf_Graph.parse("rdfGraph_smallExample.ttl", format="turtle")
    print("TTL file is valid!")

    for i, (subj, pred, obj) in enumerate(rdf_Graph):
        if i >= 10:
            break
        print(subj, pred, obj)

except Exception as e:
    print("Error in TTL file:", e)



TTL file is valid!
http://example.org/TestedMaterial http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Class
http://example.org/TestStandard http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Class
http://example.org/TestStandard http://example.org/testStandard DIN EN ISO 204:2019-4
http://example.org/hasDescription http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#DatatypeProperty
http://example.org/Compression http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://example.org/TypeOfLoading
http://example.org/digitalMaterialIdentifier http://www.w3.org/2000/01/rdf-schema#range http://www.w3.org/2001/XMLSchema#string
http://example.org/hasSpecifiedNumericValue http://www.w3.org/2000/01/rdf-schema#range http://www.w3.org/2001/XMLSchema#float
http://example.org/InitialStress http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Class
http://example.org/testID http://www.w3.org/1999/

In [3]:
# read graph as ttl
rdf_Graph=Graph()
rdf_Graph.parse("rdfGraph_smallExample.ttl", format="turtle")

<Graph identifier=N8939c7bf6d5a4b63acb5f70243a9d94f (<class 'rdflib.graph.Graph'>)>

In [4]:
# translae ttl to json-ld
jsonld_Graph = rdf_Graph.serialize(format="json-ld", indent=4)
with open("dataGraph.jsonld", "w", encoding="utf-8") as f:
    f.write(jsonld_Graph)

print("JSON-LD saved as dataGraph.jsonld.")

JSON-LD saved as dataGraph.jsonld.


In [5]:
# read graph as json-ld
jsonld_Graph=Graph()
jsonld_Graph.parse("dataGraph.jsonld", format="json-ld")

<Graph identifier=Nd8029fb7d4bd4395a1ae34857b89a1ed (<class 'rdflib.graph.Graph'>)>

In [6]:
# read schacl shapes
shacl_shape = Graph()
shacl_shape.parse("shaclShape_smallExample.ttl", format="turtle")


<Graph identifier=Nb5b2d50fa0a241d39be11e2127408811 (<class 'rdflib.graph.Graph'>)>

In [7]:
# validate data_graph agains shacl_shapes
results = validate(data_graph=rdf_Graph,
      shacl_graph=shacl_shape,
      inference='rdfs',
      data_graph_format="ttl",
    shacl_graph_format="ttl",
      abort_on_first=False,
      allow_infos=False,
      allow_warnings=False,
      meta_shacl=False,
      advanced=False,
      js=False,
      debug=True,
      serialize_report_graph="ttl")
conforms, report_graph, report_text = results
print("conforms", conforms)

Cloning DataGraph to temporary memory graph before pre-inferencing.
Running pre-inferencing with option='rdfs'.
Found 7 SHACL Shapes defined with type sh:NodeShape.
Found 0 SHACL Shapes defined with type sh:PropertyShape.
Found 0 property paths to follow.
Found 7 implied SHACL Shapes based on their properties.
Found 12 implied SHACL Shapes used as values in shape-expecting constraints.
Cached 7 unique NodeShapes and 12 unique PropertyShapes.
Validating DataGraph named N6623c5caca4a476c90873909312c6b5f
Checking if Shape <NodeShape http://example.org/shapes/InitialStress> defines its own targets.
Identifying targets to find focus nodes.
Milliseconds to find focus nodes: 0.129ms
Found 0 Focus Nodes to evaluate.
Skipping shape <NodeShape http://example.org/shapes/InitialStress> because it found no focus nodes.
Checking if Shape <NodeShape http://example.org/shapes/TestJob> defines its own targets.
Identifying targets to find focus nodes.
Milliseconds to find focus nodes: 0.162ms
Found 1 Fo

conforms False


In [8]:
print(report_text)



Validation Report
Conforms: False
Results (5):
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:dateTime ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path :dateOftestStart ]
	Focus Node: :TestJob
	Result Path: :dateOftestStart
	Message: Less than 1 values on :TestJob->:dateOftestStart
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:datatype xsd:dateTime ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path :dateOfTestEnd ]
	Focus Node: :TestJob
	Result Path: :dateOfTestEnd
	Message: Less than 1 values on :TestJob->:dateOfTestEnd
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Viola

In [9]:
report_g = Graph()
report_g.parse(data=report_graph, format="ttl", encoding="utf-8")
nm = report_g.namespace_manager

for s, p, o in sorted(report_g):
    print(s.n3(nm), p.n3(nm), o.n3(nm))

_:n57823020de01402d853b77dabd2453ceb1 rdf:type sh:ValidationReport
_:n57823020de01402d853b77dabd2453ceb1 sh:conforms "false"^^xsd:boolean
_:n57823020de01402d853b77dabd2453ceb1 sh:result _:n57823020de01402d853b77dabd2453ceb2
_:n57823020de01402d853b77dabd2453ceb1 sh:result _:n57823020de01402d853b77dabd2453ceb4
_:n57823020de01402d853b77dabd2453ceb1 sh:result _:n57823020de01402d853b77dabd2453ceb5
_:n57823020de01402d853b77dabd2453ceb1 sh:result _:n57823020de01402d853b77dabd2453ceb7
_:n57823020de01402d853b77dabd2453ceb1 sh:result _:n57823020de01402d853b77dabd2453ceb8
_:n57823020de01402d853b77dabd2453ceb10 rdf:first :Compression
_:n57823020de01402d853b77dabd2453ceb10 rdf:rest _:n57823020de01402d853b77dabd2453ceb11
_:n57823020de01402d853b77dabd2453ceb11 rdf:first :Bending
_:n57823020de01402d853b77dabd2453ceb11 rdf:rest rdf:nil
_:n57823020de01402d853b77dabd2453ceb12 rdf:first :Tension
_:n57823020de01402d853b77dabd2453ceb12 rdf:rest _:n57823020de01402d853b77dabd2453ceb10
_:n57823020de01402d853b7